In [7]:
import os, sys
sys.path.append("/usr/local/google/home/stellasyan/Documents/llm_internalization/")

import pandas as pd
from utils import bagz_utils
import config
import numpy as np

In [2]:
df = bagz_utils.read_parquet(config.META_W_SID)
df.head(3)

,asin,description,title,imUrl,salesRank,categories,price,related,brand,IID,fine_category,formatted_text,embedding,llama_embedding,sid,formatted_sid
0,7806397051,An extensive range of 15 multiple vibrant long...,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,http://ecx.images-amazon.com/images/I/41Rn18Oe...,"{'Arts, Crafts & Sewing': None, 'Baby': None, ...","[[Beauty, Makeup, Face, Concealers & Neutraliz...",5.04,"{'also_bought': ['B00KR26VFE', 'B00E7LQHZ0', '...",COKA,1,Concealers & Neutralizers,COKA\nConcealers & Neutralizers\n5.04\nWAWO 15...,"[0.02441297098994255, -0.05267857760190964, 0....","[-0.005702972412109375, 0.0272369384765625, 0....","[1, 58, 120, 0]",A1 B58 C120 D0
1,9759091062,Xtreme Brite Brightening gel is a highly conc...,Xtreme Brite Brightening Gel 1oz.,http://ecx.images-amazon.com/images/I/41QWW9v1...,"{'Arts, Crafts & Sewing': None, 'Baby': None, ...","[[Beauty, Hair Care, Styling Products, Creams,...",19.99,"{'also_bought': ['B0054GLD1U', 'B003BRZCUC', '...",Xtreme Brite,2,"Creams, Gels & Lotions","Xtreme Brite\nCreams, Gels & Lotions\n19.99\nX...","[0.0014833217719569802, -0.10309720784425735, ...","[-0.0054779052734375, -0.0028934478759765625, ...","[22, 90, 249, 0]",A22 B90 C249 D0
2,9788072216,Prada Candy By Prada Eau De Parfum Spray 1.7 O...,Prada Candy By Prada Eau De Parfum Spray 1.7 O...,http://ecx.images-amazon.com/images/I/51iT2k6L...,"{'Arts, Crafts & Sewing': None, 'Baby': None, ...","[[Beauty, Fragrance, Women's, Eau de Parfum]]",65.86,"{'also_bought': ['B006C5OHSI', 'B006P14842', '...",Prada,3,Eau de Parfum,Prada\nEau de Parfum\n65.86\nPrada Candy By Pr...,"[0.07040080428123474, -0.013543113134801388, -...","[-0.01143646240234375, 0.028839111328125, 0.01...","[24, 147, 221, 0]",A24 B147 C221 D0


### Check the statistics of the first level sid

In [13]:
# Extract the first element of 'sid'
first_elements = df['sid'].apply(lambda x: x[0])

# Get statistics
stats = first_elements.describe()   # count, mean, std, min, 25%, 50%, 75%, max
value_counts = first_elements.value_counts()  # frequency of each first element

print("Descriptive statistics:\n", stats)
print("\nValue counts:\n", value_counts)

Descriptive statistics:
 count    12101.000000
mean       107.871746
std         72.591989
min          0.000000
25%         46.000000
50%         93.000000
75%        161.000000
max        255.000000
Name: sid, dtype: float64

Value counts:
 sid
147    315
78     245
161    229
108    224
81     192
      ... 
201     10
26       9
19       9
107      5
117      1
Name: count, Length: 186, dtype: int64


### Check prefix statistics

In [8]:
def filter_by_prefix(df, prefix):
    n = len(prefix)
    prefix = np.array(prefix)  # make sure it's an array
    # Use all() along axis 1 to get a single True/False
    return df[df['sid'].apply(lambda x: np.all(x[:n] == prefix))]


In [22]:
result = filter_by_prefix(df, [78])
# print(result)

stats = result['fine_category'].value_counts()      # Count of each unique category
percent = result['fine_category'].value_counts(normalize=True) * 100  # Percentage

print("Counts:\n", stats)
print("\nPercentages:\n", percent)

Counts:
 fine_category
Mascara                       228
Eyeliner                        4
Body Scrubs                     3
Eyebrow Color                   2
Fake Eyelashes & Adhesives      1
Eyelash Tools                   1
Creams & Moisturizers           1
Bath Mitts & Cloths             1
Hair Styling Serums             1
Curlers                         1
Lip Liners                      1
Serums                          1
Name: count, dtype: int64

Percentages:
 fine_category
Mascara                       93.061224
Eyeliner                       1.632653
Body Scrubs                    1.224490
Eyebrow Color                  0.816327
Fake Eyelashes & Adhesives     0.408163
Eyelash Tools                  0.408163
Creams & Moisturizers          0.408163
Bath Mitts & Cloths            0.408163
Hair Styling Serums            0.408163
Curlers                        0.408163
Lip Liners                     0.408163
Serums                         0.408163
Name: proportion, dtype: float64


In [18]:
# Function to get statistics of nth element given a filter on the first element
def stats_second_given_first(df, first_value):
    # Filter rows where first element matches
    filtered = df[df['sid'].apply(lambda x: x[0] == first_value)]
    # Extract second element
    second_elements = filtered['sid'].apply(lambda x: x[1])
    # Descriptive statistics
    desc_stats = second_elements.describe()
    # Value counts
    counts = second_elements.value_counts()
    return desc_stats, counts

# Example: first element == 32
desc_stats, counts = stats_second_given_first(df, 147)

print("Descriptive statistics of 2nd element:\n", desc_stats)
print("\nValue counts of 2nd element:\n", counts)

Descriptive statistics of 2nd element:
 count    315.000000
mean     134.726984
std       79.589202
min        1.000000
25%       59.000000
50%      129.000000
75%      218.000000
max      254.000000
Name: sid, dtype: float64

Value counts of 2nd element:
 sid
246    17
57     17
224    16
218    14
114     9
       ..
126     1
186     1
44      1
22      1
142     1
Name: count, Length: 116, dtype: int64


In [19]:
result3 = filter_by_prefix(df, [147, 246])
# print(result)

stats = result3['fine_category'].value_counts()      # Count of each unique category
percent = result3['fine_category'].value_counts(normalize=True) * 100  # Percentage

print("Counts:\n", stats)
print("\nPercentages:\n", percent)

Counts:
 fine_category
Nail Polish    17
Name: count, dtype: int64

Percentages:
 fine_category
Nail Polish    100.0
Name: proportion, dtype: float64


In [ ]:
result3 = filter_by_prefix(df, [147, 57])
# print(result)

stats = result3['fine_category'].value_counts()      # Count of each unique category
percent = result3['fine_category'].value_counts(normalize=True) * 100  # Percentage

print("Counts:\n", stats)
print("\nPercentages:\n", percent)

Counts:
 brand
China Glaze    17
Name: count, dtype: int64

Percentages:
 fine_category
Nail Polish    100.0
Name: proportion, dtype: float64
